## **Notebook 03: Tool Integration and OpenAI Function Calling**

### **Setup**

In [2]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

print("✅ Imports successful!")
print("🤖 Ready to build AI agents!")

✅ Imports successful!
🤖 Ready to build AI agents!


In [5]:
# Verify LangGraph version
import importlib.metadata as md
print(md.version("langgraph"))

1.2.9


### **Understanding Message State**

In [6]:
# State for agents - note the Annotated type with add_messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [8]:
# Let's test how add_messages works
from langgraph.graph.message import add_messages

# Existing messages
existing = [HumanMessage(content="Hello")]

# New messages
new = [AIMessage(content="Hi there!")]

result = add_messages(existing, new)

In [9]:
print("Existing messages:", len(existing))
print("New messages:", len(new))
print("Combined messages:", len(result))
print("\nMessages:")
for msg in result:
    print(f"  {type(msg).__name__}: {msg.content}")

Existing messages: 1
New messages: 1
Combined messages: 2

Messages:
  HumanMessage: Hello
  AIMessage: Hi there!


### **🆕 Built-in MessagesState**

In [10]:
# New: use the built-in MessagesState
from langgraph.graph import MessagesState

# Old way (still valid):
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

### **Simple Weather Agent**

##### Step 1: Define Tools

In [11]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
        
    Returns:
        A string describing the current weather.
    """
    weather_data = {
        "paris": "Sunny, 22°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C",
    }

    city_lower = city.lower()
    if city_lower in weather_data:
        return f"The weather in {city} is {weather_data[city_lower]}"
    else:
        return f"Sorry, I don't have weather data for {city}"
    
# Test the tool directly
print("Testing tool:")
print(get_weather.invoke({"city":"Paris"}))

Testing tool:
The weather in Paris is Sunny, 22°C


##### Step 2: Create LLM with Tools